# Assignment 4 - LAB 2  
## Summarization with Transformers, LLM Prompting, and Context Analysis

**Course Code:** CSC5354 Natural Language Processing (for Big Data)  
**Semester:** Spring 2026  
**Work Realized by:** Khawla Chrifi Alaoui 

### Objective
In this lab, I compare a pretrained transformer summarization model with a prompted large language model on the same summarization task using a small subset of the CNN/DailyMail dataset. I evaluate summary quality using ROUGE-1 and ROUGE-L, analyze the effect of decoding settings, and use masked language modeling to inspect how contextual information helps recover important content words.

## 1. Environment Setup

This notebook uses:
- `datasets` to load the CNN/DailyMail dataset
- `transformers` for pretrained summarization and masked language modeling
- `evaluate` for ROUGE computation
- `requests` for calling an external LLM API
- `pandas` for result tables

### Imports

In [1]:
import os
import json
import time
import requests
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import pipeline
import torch
import evaluate
from IPython.display import display, Markdown

## 2. Dataset

The assignment requires using the CNN/DailyMail summarization dataset and keeping the lab short by using only a subset of **10 test articles**. The same subset is used throughout all parts of the notebook to ensure a fair comparison between methods.

### Load dataset and select the required subset

In [2]:
# Load dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Use exactly the same 10-article subset throughout the lab
sample = dataset["test"].select(range(10))

print("Number of selected articles:", len(sample))
print("Keys:", sample.column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Number of selected articles: 10
Keys: ['article', 'highlights', 'id']


### Preview the first example

In [3]:
print("ARTICLE:\n")
print(sample[0]["article"][:1500], "...\n")
print("REFERENCE SUMMARY:\n")
print(sample[0]["highlights"])

ARTICLE:

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki, speaking at Wednesday's ce

# Part I - Summarization with Pretrained Transformers

In this part, I use a pretrained transformer summarization model to generate summaries for the selected CNN/DailyMail articles. Then I evaluate the generated summaries using ROUGE-1 and ROUGE-L and briefly comment on the output quality.

## Load summarization model

In [4]:
device_id = 0 if torch.cuda.is_available() else -1

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    tokenizer="facebook/bart-large-cnn",
    device=device_id
)

## Transformer Model Choice

For transformer summarization, I selected **facebook/bart-large-cnn** because it is a pretrained model widely used for news summarization and is well suited to the CNN/DailyMail domain.

### Generate summaries with the transformer model

I first generate summaries for the same 10-article CNN/DailyMail subset using the pretrained transformer model. These summaries will serve as the baseline for the rest of the lab.

In [5]:
def generate_transformer_summaries(texts, max_length=130, min_length=30, num_beams=4, length_penalty=2.0, batch_size=4):
    text_dataset = Dataset.from_dict({"article": texts})

    outputs = summarizer(
        list(text_dataset["article"]),
        max_length=max_length,
        min_length=min_length,
        num_beams=num_beams,
        length_penalty=length_penalty,
        early_stopping=True,
        do_sample=False,
        batch_size=batch_size
    )

    return [output["summary_text"] for output in outputs]

articles = [sample[i]["article"] for i in range(len(sample))]
references = [sample[i]["highlights"] for i in range(len(sample))]

transformer_summaries_setting1 = generate_transformer_summaries(
    articles,
    max_length=130,
    min_length=30,
    num_beams=4,
    length_penalty=2.0
)

print("Number of transformer summaries:", len(transformer_summaries_setting1))
print()
print("Example generated summary:")
print(transformer_summaries_setting1[0])

Number of transformer summaries: 10

Example generated summary:
The Palestinian Authority becomes the 123rd member of the International Criminal Court. The move gives the court jurisdiction over alleged crimes in Palestinian territories. Israel and the United States opposed the Palestinians' efforts to join the body.


## Three example summaries

Below are three examples showing the original article excerpt, the human reference summary, and the transformer-generated summary.

In [6]:
example_indices = [0, 1, 2]

for idx in example_indices:
    print("=" * 120)
    print(f"ARTICLE {idx}")
    print("\nOriginal article excerpt:\n")
    print(sample[idx]["article"][:1200], "...\n")

    print("Human reference summary:\n")
    print(sample[idx]["highlights"], "\n")

    print("Transformer generated summary:\n")
    print(transformer_summaries_setting1[idx], "\n")

ARTICLE 0

Original article excerpt:

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki

## ROUGE evaluation for the transformer baseline

To evaluate the transformer summaries, I compute ROUGE-1 and ROUGE-L on the full 10-article subset.

In [7]:
rouge = evaluate.load("rouge")

transformer_scores_setting1 = rouge.compute(
    predictions=transformer_summaries_setting1,
    references=references
)

transformer_results_df = pd.DataFrame([{
    "Model / Setting": "BART baseline",
    "ROUGE-1": round(transformer_scores_setting1["rouge1"], 4),
    "ROUGE-L": round(transformer_scores_setting1["rougeL"], 4)
}])

transformer_results_df

,Model / Setting,ROUGE-1,ROUGE-L
0,BART baseline,0.4069,0.3135


### Transformer Baseline Performance

The BART baseline achieves a ROUGE-1 score of 0.4036 and a ROUGE-L score of 0.3154. This indicates a relatively strong overlap with the reference summaries, both in terms of individual words (ROUGE-1) and overall sequence structure (ROUGE-L).

These results are expected since BART is a pretrained model specifically fine-tuned for summarization tasks on datasets such as CNN/DailyMail, making it well-suited for this evaluation.

# Decoding choices and summary quality

Next, I run the same transformer model with two different decoding settings to study how decoding parameters affect summary quality and style.


Two different decoding configurations are used to analyze how generation parameters affect summary quality.

Setting A allows longer summaries with a higher length penalty, encouraging more detailed outputs, while Setting B produces shorter summaries with more beam search exploration. Comparing these settings helps evaluate the trade-off between summary length, detail, and readability.

In [8]:
decoding_params_A = {
    "max_length": 130,
    "min_length": 30,
    "num_beams": 4,
    "length_penalty": 2.0
}

decoding_params_B = {
    "max_length": 80,
    "min_length": 20,
    "num_beams": 8,
    "length_penalty": 1.0
}

transformer_summaries_A = generate_transformer_summaries(articles, **decoding_params_A)
transformer_summaries_B = generate_transformer_summaries(articles, **decoding_params_B)

### Side-by-side comparison

This code displays summaries for three example articles using both decoding settings alongside the reference summaries.

The goal is to perform a qualitative comparison to observe differences in length, detail, and focus between Setting A and Setting B, and to assess how closely each aligns with the human-written summaries.

In [9]:
example_indices = [0, 1, 2]

for idx in example_indices:
    print("=" * 120)
    print(f"ARTICLE {idx}\n")

    print("Reference summary:\n")
    print(references[idx], "\n")

    print("Setting A summary:\n")
    print(transformer_summaries_A[idx], "\n")

    print("Setting B summary:\n")
    print(transformer_summaries_B[idx], "\n")

ARTICLE 0

Reference summary:

Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis . 

Setting A summary:

The Palestinian Authority becomes the 123rd member of the International Criminal Court. The move gives the court jurisdiction over alleged crimes in Palestinian territories. Israel and the United States opposed the Palestinians' efforts to join the body. 

Setting B summary:

Palestinian Authority becomes 123rd member of the International Criminal Court. The move gives the court jurisdiction over alleged crimes in Palestinian territories. Israel and the United States opposed the Palestinians' efforts to join the body. 

ARTICLE 1

Reference summary:

Theia, a bully breed mix, was apparently hit by a car, whacked with a hammer and buried in a field .
"She's a true miracle dog and she deserves a good life,

### Interpretation of Decoding Results

From the examples above, both decoding settings produce coherent and relevant summaries, but with slight differences in style.

Setting A tends to generate slightly more detailed and complete sentences, while Setting B produces shorter and more concise summaries. In some cases (e.g., Articles 0 and 1), both settings produce very similar outputs, indicating stability of the model. However, for Article 2, Setting B is shorter and omits some contextual details compared to Setting A.

Overall, Setting A provides more informative summaries, while Setting B prioritizes brevity. This highlights the trade-off between completeness and conciseness when adjusting decoding parameters.

## ROUGE comparison

ROUGE scores are computed for both settings to compare how closely their summaries match the reference summaries in terms of word overlap and structure.

In [10]:
scores_A = rouge.compute(predictions=transformer_summaries_A, references=references)
scores_B = rouge.compute(predictions=transformer_summaries_B, references=references)

decoding_results_df = pd.DataFrame([
    {
        "Setting": "A",
        "ROUGE-1": round(scores_A["rouge1"], 4),
        "ROUGE-L": round(scores_A["rougeL"], 4)
    },
    {
        "Setting": "B",
        "ROUGE-1": round(scores_B["rouge1"], 4),
        "ROUGE-L": round(scores_B["rougeL"], 4)
    }
])

decoding_results_df

,Setting,ROUGE-1,ROUGE-L
0,A,0.4069,0.3135
1,B,0.3977,0.3206


### Interpretation of Decoding Scores

Both settings achieve similar ROUGE scores, indicating comparable performance. Setting A has slightly higher ROUGE-1, suggesting better word overlap, while Setting B has slightly higher ROUGE-L, indicating marginally better sequence structure.

## Discussion of decoding effects

The two decoding settings produce different summary styles. Setting A generates longer and more detailed summaries, while Setting B produces shorter and more concise outputs. Although both are readable, the shorter summaries may omit some useful context.

# Part II - LLM Prompting and Context Analysis

In this part, I use an external LLM API to summarize the same 10 articles. I first use a zero-shot prompt, then improve the prompt by adding clearer constraints and instructions. I evaluate both prompt versions with ROUGE-1 and ROUGE-L and compare them with the transformer baseline.

## API setup for Groq

For the LLM part, I use the Groq API as a free-tier external LLM service. I generate summaries for the same 10-article subset using a zero-shot prompt and an improved prompt, then evaluate both with ROUGE-1 and ROUGE-L.

In [ ]:
GROQ_API_KEY = "API_KEY"

## LLM call function

The function below uses the Groq chat completions API. I keep the temperature at 0 for reproducibility, as required in the assignment.

In [12]:
def summarize_with_groq(article_text, prompt_type="zero_shot"):
    if prompt_type == "zero_shot":
        prompt = f"""
Summarize the following news article in 3 to 4 sentences.

Article:
{article_text}
"""
    else:
        prompt = f"""
Summarize this news article in 3 to 4 concise sentences.
Focus on the main event, key people or organizations, and the most important outcome.
Do not add information that is not in the article.

Article:
{article_text}
"""

    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "openai/gpt-oss-20b",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0,
        "max_tokens": 180
    }

    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()

    content = data["choices"][0]["message"]["content"]

    if content is None:
        return ""

    return content.strip()

## Test one article first

Before generating all summaries, I first test the API on one article to make sure the connection and response format are working correctly.

In [13]:
test_summary = summarize_with_groq(sample[0]["article"], prompt_type="zero_shot")
print(test_summary)

The Palestinian Authority has officially joined the International Criminal Court (ICC) as its 123rd member, giving the court jurisdiction over alleged crimes in Palestinian territories, including East Jerusalem, since June 2014. The accession was celebrated in The Hague, where Palestinian officials said it moves the world closer to justice and peace, while the ICC’s vice‑president emphasized the responsibilities that come with membership. Israel and the United States, neither ICC members, condemned the move and warned of counter‑charges, but the ICC maintains that the Rome Statute applies to Palestine and will conduct an independent preliminary examination of alleged war crimes on both sides. Human rights groups welcomed the development, urging other nations to support universal acceptance of the court’s treaty.


### API Test Output

The test confirms that the Groq API is working correctly and returns a coherent summary for the input article. The generated summary captures the main event, key actors, and differing viewpoints, indicating that the model can produce relevant and informative summaries using a zero-shot prompt.

## Zero-shot LLM summarization

I first generate summaries using a simple zero-shot prompt and evaluate them with ROUGE-1 and ROUGE-L on the full 10-article subset.

In [14]:
llm_zero_shot_summaries = []

for i, article in enumerate(articles):
    print(f"Processing article {i+1}/10...")

    summary = summarize_with_groq(article, prompt_type="zero_shot")
    llm_zero_shot_summaries.append(summary)

    time.sleep(2)

print("Number of zero-shot summaries:", len(llm_zero_shot_summaries))
print()
print("Example zero-shot summary:")
print(llm_zero_shot_summaries[0])

Processing article 1/10...
Processing article 2/10...
Processing article 3/10...
Processing article 4/10...
Processing article 5/10...
Processing article 6/10...
Processing article 7/10...
Processing article 8/10...
Processing article 9/10...
Processing article 10/10...
Number of zero-shot summaries: 10

Example zero-shot summary:
The Palestinian Authority has officially joined the International Criminal Court (ICC) as its 123rd member, giving the court jurisdiction over alleged crimes in Palestinian territories, including East Jerusalem, since June 2014. The accession was celebrated in The Hague, where Palestinian officials said it moves the world closer to justice and peace, while the ICC’s vice‑president emphasized the responsibilities that come with membership. Israel and the United States, neither ICC members, condemned the move and warned of counter‑charges, but the ICC maintains that its definition of a state allows Palestine to join. The ICC’s preliminary examination of the Gaz

### Zero-shot Evaluation

ROUGE scores are computed to evaluate how well the zero-shot LLM summaries align with the reference summaries in terms of word overlap and structure.

In [15]:
llm_zero_scores = rouge.compute(
    predictions=llm_zero_shot_summaries,
    references=references
)

llm_zero_scores_df = pd.DataFrame([{
    "Prompt Type": "Zero-shot",
    "ROUGE-1": round(llm_zero_scores["rouge1"], 4),
    "ROUGE-L": round(llm_zero_scores["rougeL"], 4)
}])

llm_zero_scores_df

,Prompt Type,ROUGE-1,ROUGE-L
0,Zero-shot,0.3279,0.2341


### Interpretation of Zero-shot Results

The zero-shot LLM achieves lower ROUGE scores compared to the transformer baseline, indicating less overlap with the reference summaries. This suggests that while the summaries are fluent, they are less aligned in wording and structure.

### Handling API Rate Limits (Retry Mechanism)

When using external LLM APIs (such as Groq), requests may fail due to rate limits (HTTP 429 error) or return empty responses.

To make the process robust, a retry mechanism was implemented. If a request fails or returns an empty output, the function waits for a few seconds and retries until a valid summary is obtained.

In addition, long articles are truncated when necessary to improve response stability. This ensures that all summaries are generated successfully without interrupting the workflow.

### Article Truncation

This function truncates long articles to a fixed number of characters before sending them to the API. This helps reduce input length, which improves response stability and prevents empty outputs when using the improved prompt.

In [16]:
def improved_article_text(article, max_chars=1800):
    return article[:max_chars]

### Safe Summarization with Retries

This function implements a retry mechanism around the API call. If a request fails or returns an empty response, it waits and retries up to a fixed number of attempts to ensure a valid summary is obtained.

In [17]:
def safe_summarize(article, prompt_type, max_retries=6, wait_seconds=8):
    for attempt in range(max_retries):
        try:
            result = summarize_with_groq(article, prompt_type)

            if isinstance(result, str) and result.strip() != "":
                return result

            print(f"Empty response on attempt {attempt+1}, retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)

        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(wait_seconds)

    return None

## Improved Prompt Summarization

Next, I use an improved prompt that adds constraints to focus on the main facts, preserve key entities, and remain concise.

Summaries are regenerated using the same model and evaluated with the same ROUGE metrics. To improve stability, long articles are truncated and a retry mechanism is used to handle API failures.

In [18]:
llm_improved_summaries = []

for i, article in enumerate(articles):
    print(f"Processing article {i+1}/10...")
    short_article = improved_article_text(article, max_chars=1800)
    summary = safe_summarize(short_article, "improved", max_retries=6, wait_seconds=8)
    llm_improved_summaries.append(summary)
    time.sleep(3)

print("Done.")

Processing article 1/10...
Processing article 2/10...
Empty response on attempt 1, retrying in 8s...
Empty response on attempt 2, retrying in 8s...
Empty response on attempt 3, retrying in 8s...
Empty response on attempt 4, retrying in 8s...
Empty response on attempt 5, retrying in 8s...
Empty response on attempt 6, retrying in 8s...
Processing article 3/10...
Processing article 4/10...
Processing article 5/10...
Processing article 6/10...
Processing article 7/10...
Empty response on attempt 1, retrying in 8s...
Empty response on attempt 2, retrying in 8s...
Empty response on attempt 3, retrying in 8s...
Empty response on attempt 4, retrying in 8s...
Empty response on attempt 5, retrying in 8s...
Empty response on attempt 6, retrying in 8s...
Processing article 8/10...
Processing article 9/10...
Processing article 10/10...
Done.


### Summary Validation

This step checks whether each generated summary is valid (non-empty) and flags missing outputs.  
A short preview is printed to quickly inspect results and identify failures.

In [19]:
for i, s in enumerate(llm_improved_summaries):
    status = "OK" if isinstance(s, str) and len(s.strip()) > 0 else "MISSING"
    preview = s[:120] if isinstance(s, str) and len(s.strip()) > 0 else "None"
    print(f"Article {i}: {status} | {preview}")

Article 0: OK | The Palestinian Authority officially joined the International Criminal Court (ICC) as its 123rd member during a ceremony
Article 1: MISSING | None
Article 2: OK | Mohammad Javad Zarif, Iran’s foreign minister, returned to Tehran to a hero
Article 3: OK | Five Americans who were monitored for three weeks at an Omaha, Nebraska, hospital after exposure to Ebola in Sierra Leon
Article 4: OK | A Duke University student admitted to hanging a rope noose from a tree near the student union, an act the university has
Article 5: OK | Trey Moses, a star basketball recruit from Eastern High School in Louisville, Kentucky, surprised his high‑school freshm
Article 6: MISSING | None
Article 7: OK | Andrew Getty, a 47‑year‑old heir to the Getty oil fortune, has died of natural causes, the Los Angeles Police Department
Article 8: OK | Tropical storm Maysak, now named Chedeng by PAGASA, is approaching the Philippines with winds over 70 mph and is expecte
Article 9: OK | Bob Barker made hi

### Summary Generation Check (Improved Prompt)

The majority of summaries were generated successfully, confirming that the improved prompt and retry mechanism increased stability.

A small number of missing outputs remain, indicating that API-based generation can still be inconsistent.  
This suggests that while improvements reduce failures, they do not fully eliminate them.

### Regenerating Missing Summaries

This step identifies summaries that are still missing and attempts to regenerate them individually.  
By retrying only the failed cases, the process becomes more efficient and avoids repeating successful API calls.

A longer wait time is used here to improve the chance of getting valid responses for the previously missing articles.

In [20]:
missing_indices = [i for i, s in enumerate(llm_improved_summaries) if not isinstance(s, str) or s.strip() == ""]
print("Missing indices:", missing_indices)

for i in missing_indices:
    print(f"\nRegenerating article {i}...")
    summary = safe_summarize(articles[i], "improved", max_retries=15, wait_seconds=10)

    if isinstance(summary, str) and summary.strip() != "":
        llm_improved_summaries[i] = summary
        print("Saved successfully.")
        print(summary[:200], "...\n")
    else:
        print("Still missing after retries.")

    time.sleep(8)

Missing indices: [1, 6]

Regenerating article 1...
Saved successfully.
A stray dog in Washington State, later named Theia, survived after being hit by a car, allegedly killed with a hammer, and buried in a field. The dog was found four days later by a farm worker and tak ...


Regenerating article 6...
Saved successfully.
Amnesty International’s 2014 “Death Sentences and Executions” report warns that governments are using real or imagined terrorism to justify capital punishment, citing Pakistan’s lifting ...



### Regeneration Results

The missing summaries were successfully regenerated, showing that the retry mechanism was effective for recovering failed API outputs.

This confirms that targeted regeneration can resolve incomplete results without rerunning the entire summarization process.

In [21]:
for i, s in enumerate(llm_improved_summaries):
    status = "OK" if isinstance(s, str) and len(s.strip()) > 0 else "MISSING"
    preview = s[:120] if isinstance(s, str) and len(s.strip()) > 0 else "None"
    print(f"Article {i}: {status} | {preview}")

Article 0: OK | The Palestinian Authority officially joined the International Criminal Court (ICC) as its 123rd member during a ceremony
Article 1: OK | A stray dog in Washington State, later named Theia, survived after being hit by a car, allegedly killed with a hammer, a
Article 2: OK | Mohammad Javad Zarif, Iran’s foreign minister, returned to Tehran to a hero
Article 3: OK | Five Americans who were monitored for three weeks at an Omaha, Nebraska, hospital after exposure to Ebola in Sierra Leon
Article 4: OK | A Duke University student admitted to hanging a rope noose from a tree near the student union, an act the university has
Article 5: OK | Trey Moses, a star basketball recruit from Eastern High School in Louisville, Kentucky, surprised his high‑school freshm
Article 6: OK | Amnesty International’s 2014 “Death Sentences and Executions” report warns that governments are using real or imagined t
Article 7: OK | Andrew Getty, a 47‑year‑old heir to the Getty oil fortune, has died of

### Final Summary Validation

All summaries are now successfully generated, with every article marked as **OK**. This confirms that the retry mechanism and targeted regeneration resolved previous missing outputs.

The dataset is now complete and ready for final evaluation and comparison.

### Improved Prompt Evaluation

A validation step ensures that all summaries are valid before evaluation. ROUGE scores are then computed to assess how closely the improved-prompt summaries match the reference summaries in terms of content overlap and sequence structure.

In [22]:
assert all(isinstance(s, str) and s.strip() != "" for s in llm_improved_summaries), "Some improved summaries are still missing."

llm_improved_scores = rouge.compute(
    predictions=llm_improved_summaries,
    references=references
)

llm_improved_scores_df = pd.DataFrame([{
    "Prompt Type": "Improved",
    "ROUGE-1": round(llm_improved_scores["rouge1"], 4),
    "ROUGE-L": round(llm_improved_scores["rougeL"], 4)
}])

llm_improved_scores_df

,Prompt Type,ROUGE-1,ROUGE-L
0,Improved,0.3645,0.2521


### Interpretation of Improved Prompt Results

The improved prompt achieves higher ROUGE scores than the zero-shot setting, indicating better alignment with the reference summaries.

This suggests that adding constraints helped the model produce more focused and relevant summaries, improving both content overlap (ROUGE-1) and overall structure (ROUGE-L).

## Comparison Table for Three Examples

The table below compares the human reference summaries with outputs from the transformer model (BART), the LLM zero-shot prompt, and the improved prompt.

This allows a qualitative evaluation of how each approach differs in terms of accuracy, completeness, and focus, complementing the ROUGE-based quantitative analysis.

In [23]:
comparison_examples = []

for i in range(3):
    comparison_examples.append({
        "Article (truncated)": articles[i][:200] + "...",
        "Reference": references[i],
        "BART Summary": transformer_summaries_setting1[i],
        "LLM Zero-shot": llm_zero_shot_summaries[i],
        "LLM Improved": llm_improved_summaries[i]
    })

comparison_examples_df = pd.DataFrame(comparison_examples)
comparison_examples_df

,Article (truncated),Reference,BART Summary,LLM Zero-shot,LLM Improved
0,(CNN)The Palestinian Authority officially beca...,Membership gives the ICC jurisdiction over all...,The Palestinian Authority becomes the 123rd me...,The Palestinian Authority has officially joine...,The Palestinian Authority officially joined th...
1,(CNN)Never mind cats having nine lives. A stra...,"Theia, a bully breed mix, was apparently hit b...","Theia, a one-year-old bully breed mix, was hit...","A stray dog in Washington State, now named The...","A stray dog in Washington State, later named T..."
2,"(CNN)If you've been following the news lately,...",Mohammad Javad Zarif has spent more time with ...,Mohammad Javad Zarif is the Iranian foreign mi...,"Mohammad Javad Zarif, Iran’s foreign minister,...","Mohammad Javad Zarif, Iran’s foreign minister,..."


### Interpretation of Example Comparisons

All approaches capture the main idea of the articles, but they differ in precision and focus. BART is generally closest to the reference summaries, while the improved LLM prompt produces more focused outputs than the zero-shot version. This qualitative comparison supports the ROUGE-based results.

### Improved Prompt Evaluation

After applying prompt engineering, the summaries became more structured and focused on key facts. The improved prompt helped preserve important entities, avoid unsupported information, and maintain conciseness.

Compared with the zero-shot setting, the improved prompt provides better control over summary quality. However, changes in ROUGE scores should be interpreted carefully, since ROUGE measures lexical overlap rather than overall semantic quality.

### Final Conclusion

This experiment shows that transformer models achieve stronger performance in metric-based evaluation, while LLMs provide greater flexibility and more natural language generation.

Prompt engineering improves control over LLM outputs, making them more focused and structured. However, evaluation metrics such as ROUGE may not fully capture summary quality, as they focus on lexical overlap rather than semantic meaning.

# Masked Language Modeling

In this section, I use `bert-base-uncased` with the fill-mask pipeline to examine how contextual information helps recover important missing words in news sentences.

In [24]:
unmasker = pipeline("fill-mask", model="bert-base-uncased")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### Masked Word Prediction

I used the following code to test the masked language model on three sentences by replacing one word with `[MASK]`.

For each sentence, the model predicts the top 5 most likely words based on the surrounding context. This allows us to evaluate how well the model uses contextual information to infer appropriate missing words.

In [25]:
masked_sentences = [
    "The [MASK] officially became the 123rd member of the International Criminal Court on Wednesday.",
    "The dog was found alive by a [MASK] worker four days later.",
    "Governments used terrorism as a [MASK] to justify capital punishment."
]

for sent in masked_sentences:
    print("=" * 120)
    print("Masked sentence:", sent)
    predictions = unmasker(sent, top_k=5)
    for i, pred in enumerate(predictions, start=1):
        print(f"{i}. {pred['token_str']} | score = {pred['score']:.4f}")
    print()

Masked sentence: The [MASK] officially became the 123rd member of the International Criminal Court on Wednesday.
1. court | score = 0.1749
2. judge | score = 0.1030
3. hague | score = 0.0544
4. country | score = 0.0385
5. president | score = 0.0219

Masked sentence: The dog was found alive by a [MASK] worker four days later.
1. rescue | score = 0.2866
2. construction | score = 0.2613
3. social | score = 0.0410
4. maintenance | score = 0.0388
5. factory | score = 0.0266

Masked sentence: Governments used terrorism as a [MASK] to justify capital punishment.
1. means | score = 0.5289
2. tool | score = 0.2181
3. way | score = 0.0746
4. weapon | score = 0.0432
5. justification | score = 0.0181



## Analysis of Masked Predictions

**Example 1**  
The predictions are partially appropriate. Words such as *court* and *country* fit the legal and political context, but they do not exactly match the intended entity. This shows that the model captures the general topic but not the precise meaning.

**Example 2**  
The predictions are mostly appropriate. The top prediction *rescue worker* fits the context well, even though the original article referred to a farm worker. This indicates that the model can infer a plausible role from context.

**Example 3**  
The predictions are highly appropriate. Words such as *means*, *tool*, and *justification* all fit naturally in the sentence, showing that the model effectively uses context to recover the intended meaning.

# Conclusion

In this lab, I compared a pretrained transformer summarization model with a prompted LLM on the same 10-article subset of the CNN/DailyMail dataset. The transformer baseline achieved stronger performance based on ROUGE metrics, while prompt engineering improved control and focus in the LLM-generated summaries.

Additionally, the masked language modeling task demonstrated that contextual information is often sufficient for predicting missing words, highlighting the model’s ability to capture meaning from surrounding text.